# 04 — Extract Topology Graph

Pipeline: detected bounding boxes → line detection → OCR labels → NetworkX graph.

Steps:
1. Run YOLOv8 to locate all network devices
2. Detect connector lines via Hough transform
3. Snap line endpoints to device centres
4. Extract text labels with Tesseract OCR
5. Build and visualise the topology graph

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import json
import random
from pathlib import Path
from ultralytics import YOLO

from src.topology.line_detection import detect_lines, filter_lines_by_bbox
from src.topology.graph_builder  import build_graph, graph_to_dict
from src.topology.ocr_extractor  import extract_all_labels
from src.utils.visualization     import draw_detections, plot_topology_graph

WEIGHTS   = '../outputs/infragraph_run/weights/best.pt'
IMG_DIR   = Path('../data_generator/infragraph_dataset/images/test')
GRAPH_DIR = Path('../data_generator/infragraph_dataset/graphs/test')

model     = YOLO(WEIGHTS)
img_paths = sorted(IMG_DIR.glob('*.png'))
img_path  = random.choice(img_paths)
print('Processing:', img_path.name)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

CLASS_NAMES = ['router','switch','firewall','server','database','load_balancer','cloud_or_wan']

image = cv2.imread(str(img_path))
preds = model.predict(str(img_path), imgsz=1400, conf=0.25, verbose=False)[0]

detections = []
for i, box in enumerate(preds.boxes):
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    cid = int(box.cls[0])
    detections.append({
        'id':         f'node_{i:02d}',
        'class_name': CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid),
        'confidence': float(box.conf[0]),
        'bbox':       (x1, y1, x2, y2),
    })

annotated = draw_detections(image, detections)
plt.figure(figsize=(16, 10))
plt.imshow(annotated[..., ::-1])
plt.axis('off')
plt.title(f'Detections — {img_path.name}')
plt.show()
print(f'Found {len(detections)} devices.')

In [ ]:
bboxes = [d['bbox'] for d in detections]
lines  = detect_lines(image)
lines  = filter_lines_by_bbox(lines, bboxes, margin=20)
print(f'Connector lines found: {len(lines)}')

G = build_graph(detections, lines)
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
# Compare against ground-truth graph
gt_path = GRAPH_DIR / (img_path.stem + '.json')
if gt_path.exists():
    gt = json.loads(gt_path.read_text())
    print(f'Ground truth: {len(gt["nodes"])} nodes, {len(gt["edges"])} edges')

fig = plot_topology_graph(G, title=f'Extracted Topology — {img_path.stem}')
plt.show()